In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.multioutput import MultiOutputClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.inspection import permutation_importance
import os
import fs
import process_data as dp
import mdp_utils
from scipy import stats
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

Failed to read module file 'C:\Python311\Lib\pydoc_data\topics.py' for module 'pydoc_data.topics': UnicodeDecodeError
Traceback (most recent call last):
  File "c:\Users\shirl\Documents\Studie\2025-2026\Thesis\personalized-coping-challenges\.venv\Lib\site-packages\IPython\core\extensions.py", line 62, in load_extension
    return self._load_extension(module_str)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\shirl\Documents\Studie\2025-2026\Thesis\personalized-coping-challenges\.venv\Lib\site-packages\IPython\core\extensions.py", line 77, in _load_extension
    mod = import_module(module_str)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Python311\Lib\importlib\__init__.py", line 126, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1206, in _gcd_import
  File "<frozen importlib._bootstrap>", line 1178, in _find_and_load
  File "<fr

In [2]:
data_folder = 'C:\\Users\\shirl\\Documents\\Studie\\2025-2026\\Thesis\\personalized-coping-challenges\\data'
results_folder = 'C:\\Users\\shirl\\Documents\\Studie\\2025-2026\\Thesis\\personalized-coping-challenges\\results\\'

num_clusters = 3
num_actions = num_clusters * 2
max_count = 0

# Load data
action_data = pd.read_csv(os.path.join(data_folder, 'challenge_info.csv'))
samples = pd.read_csv(os.path.join(data_folder, 'processed_samples.csv'))

cluster_vars = ['likedness', 'usefulness', 'difficulty']
actions_clustered, _, cluster_cols = dp.cluster_actions(action_data, cluster_vars, num_clusters=num_clusters)

cluster_col = 'cluster_all'
action_col = 'joint_cluster'

reward_cols = ["r_likedness", "r_usefulness", "r_expert", "r_diversity", "r_return"]
weights = [1/len(reward_cols)] * len(reward_cols)  # Equal weights for all objectives

The features we want to choose from

In [3]:
possible_state_features = ['TIR', 'TIME_Q', 'GOOD', 'MOT']


In [16]:
# We want to test multiple weight combinations
np.random.seed(66)  
weights_list = np.random.dirichlet(np.ones(len(reward_cols)), size=200)  

In [22]:
fixed_features = ['TIR', 'TIME_Q']
cluster_sizes = [2, 3, 4]
for cluster_size in cluster_sizes:
    print(f"Testing with cluster size: {cluster_size}")
    actions_clustered, _, cluster_cols = dp.cluster_actions(action_data, cluster_vars, num_clusters=cluster_size)
    num_actions = cluster_size * 2

    num_vals_per_feature = [2, 2, 2]
    print(f"     Testing with num_vals_per_feature: {num_vals_per_feature}")
    selected_with_fixed_multiple_weights = fs.feature_selection_with_fixed_multiple_weights(samples, actions_clustered, possible_state_features, fixed_features,
                                                        reward_cols, cluster_col, action_col, weights_list=weights_list, num_act=num_actions, num_vals_per_selected_feature=num_vals_per_feature,
                                                        discount_factor=0.7, scalarization='linear', max_count=max_count)
    
    num_vals_per_feature = [3, 3, 3]
    print(f"     Testing with num_vals_per_feature: {num_vals_per_feature}")
    selected_with_fixed_multiple_weights = fs.feature_selection_with_fixed_multiple_weights(samples, actions_clustered, possible_state_features, fixed_features,
                                                            reward_cols, cluster_col, action_col, weights_list=weights_list, num_act=num_actions, num_vals_per_selected_feature=num_vals_per_feature,
                                                            discount_factor=0.7, scalarization='linear', max_count=max_count)
    print()

Testing with cluster size: 2
     Testing with num_vals_per_feature: [2, 2, 2]
Candidate avg p-values: [np.float64(0.12463451056739999), np.float64(0.12293777839393882)]
Candidate avg F-stats: [np.float64(17.371052303333926), np.float64(12.979192224511461)]
Added: MOT (Avg p-value: 0.12293777839393882)
     Testing with num_vals_per_feature: [3, 3, 3]
Candidate avg p-values: [np.float64(0.04689796234949196), np.float64(0.06163307067928837)]
Candidate avg F-stats: [np.float64(29.298332830614964), np.float64(21.357973659749142)]
Added: GOOD (Avg p-value: 0.04689796234949196)

Testing with cluster size: 3
     Testing with num_vals_per_feature: [2, 2, 2]
Candidate avg p-values: [np.float64(0.09608763590261471), np.float64(0.08421243389841372)]
Candidate avg F-stats: [np.float64(20.43182205115177), np.float64(17.31240233474384)]
Added: MOT (Avg p-value: 0.08421243389841372)
     Testing with num_vals_per_feature: [3, 3, 3]
Candidate avg p-values: [np.float64(0.03557673462181319), np.float6

In [21]:
cluster_sizes = [2, 4]
for cluster_size in cluster_sizes:
    print(f"Testing with cluster size: {cluster_size}")
    actions_clustered, _, cluster_cols = dp.cluster_actions(action_data, cluster_vars, num_clusters=cluster_size)
    num_actions = cluster_size * 2
    binning_combinations = [[2,2,2], [2,2,3], [2, 3, 2], [2, 3, 3], [3, 2, 2], [3, 2, 3], [3, 3, 2], [3, 3, 3]]

    optimal_binning, results = fs.bin_selection_manual_combinations(
        df=samples,
        actions_clustered=actions_clustered,
        selected_features=['TIR', 'TIME_Q', 'GOOD'],
        reward_cols=reward_cols,
        cluster_col=cluster_col,
        action_col=action_col,
        num_act=num_actions,
        binning_combinations=binning_combinations,  # or binning_combinations_dict
        weights_list=weights_list,
        max_count=max_count,
        discount_factor=0.7,
        scalarization='linear',
        seed=42,
        min_samples_per_state=10
    )
    print()

Testing with cluster size: 2
Testing 8 binning combinations
Features (in order): ['TIR', 'TIME_Q', 'GOOD']
Using 200 weight vectors


COMBINATION 1/8
Configuration: {'TIR': 2, 'TIME_Q': 2, 'GOOD': 2}
State space size: 8
Coverage: 100.00% (8/8 states)
Samples per state: 388.9
Min samples per (s,a) pair: 8

Results:
  Mean F-stat: 6.43 ± 10.79
  Mean p-value: 0.38 ± 0.16
  Mean return: 1.5205 ± 0.8523
P-values by feature:
  TIR: 0.73 ± 0.16
  TIME_Q: 0.30 ± 0.20
  GOOD: 0.12 ± 0.22

COMBINATION 2/8
Configuration: {'TIR': 2, 'TIME_Q': 2, 'GOOD': 3}
State space size: 12
Coverage: 100.00% (12/12 states)
Samples per state: 259.2
Min samples per (s,a) pair: 8

Results:
  Mean F-stat: 6.68 ± 9.36
  Mean p-value: 0.32 ± 0.16
  Mean return: 1.5379 ± 0.8427
P-values by feature:
  TIR: 0.64 ± 0.17
  TIME_Q: 0.21 ± 0.18
  GOOD: 0.09 ± 0.21

COMBINATION 3/8
Configuration: {'TIR': 2, 'TIME_Q': 3, 'GOOD': 2}
State space size: 12
Coverage: 100.00% (12/12 states)
Samples per state: 259.2
Min samples per